# TP 2 — Fine-Tuning avec LoRA (Partie 2)

**Cours : Fine-Tuning, Prompting et Agents IA — Master 2 AI-SD**
*A. Mbayandjambe Masheke*

## Objectif
Adapter un modèle (`flan-t5-small`) à une tâche précise — **classer le sentiment d'avis en français** — grâce à **LoRA** (PEFT), en n'entraînant que ~0,3 % des paramètres.

On va :
1. Tester le modèle **avant** entraînement (baseline).
2. Préparer un **mini-jeu de données** (intégré dans le code).
3. Configurer **LoRA** et regarder le nombre de paramètres entraînés.
4. Entraîner, puis comparer **avant / après**.

> **GPU recommandé** : *Exécution → Modifier le type d'exécution → GPU*.


In [1]:
# Étape 0 — Installation
# IMPORTANT : on n'upgrade PAS torch (sinon conflit avec torchvision préinstallé sur Colab).
!pip -q install -U transformers peft datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00


> **Dépannage** — Si vous voyez `RuntimeError: operator torchvision::nms does not exist`
> (conflit `torch` / `torchvision`) : **Exécution → Redémarrer la session**, puis ré-exécutez
> à partir de cette cellule (elle n'upgrade pas torch).


## Étape 1 — Charger le modèle de base et tester la baseline
`flan-t5-small` connaît mal le sentiment **en français** : c'est justement ce qu'on va corriger.

In [2]:
import torch, random, numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

random.seed(0); np.random.seed(0); torch.manual_seed(0)

nom_modele = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(nom_modele)
modele = AutoModelForSeq2SeqLM.from_pretrained(nom_modele)
device = "cuda" if torch.cuda.is_available() else "cpu"
modele.to(device)

def predire(texte):
    prompt = f"Classe le sentiment (positif ou négatif) : {texte}"
    ids = tokenizer(prompt, return_tensors="pt").to(device)
    out = modele.generate(**ids, max_new_tokens=6)
    return tokenizer.decode(out[0], skip_special_tokens=True)

tests = ["Ce produit est vraiment excellent, je recommande.",
         "Très déçu, l'appareil est tombé en panne en deux jours."]

print("=== AVANT fine-tuning ===")
for t in tests: print(f"  {t[:45]:45} -> {predire(t)}")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

=== AVANT fine-tuning ===
  Ce produit est vraiment excellent, je recomma -> Positive
  Très déçu, l'appareil est tombé en panne en d ->   


**À observer** : les réponses avant entraînement sont souvent vides, en anglais, ou incohérentes. On garde ces résultats en tête pour la comparaison.

## Étape 2 — Le jeu de données (intégré)
On écrit directement ~30 exemples étiquetés en français. En pratique, on en aurait plus, mais **la qualité prime sur la quantité**.

In [3]:
donnees = [
    ("Ce produit est excellent, j'adore.", "positif"),
    ("Livraison rapide et emballage soigné.", "positif"),
    ("Je suis très satisfait de mon achat.", "positif"),
    ("Rapport qualité-prix imbattable.", "positif"),
    ("Fonctionne parfaitement, rien à redire.", "positif"),
    ("Service client réactif et aimable.", "positif"),
    ("Un vrai plaisir à utiliser au quotidien.", "positif"),
    ("La qualité dépasse mes attentes.", "positif"),
    ("Très bon produit, je recommande vivement.", "positif"),
    ("Design élégant et finitions impeccables.", "positif"),
    ("Parfait pour offrir, la personne a adoré.", "positif"),
    ("Simple à installer et très efficace.", "positif"),
    ("Déçu, l'article est arrivé cassé.", "négatif"),
    ("Ne fonctionne pas comme annoncé.", "négatif"),
    ("Mauvaise qualité, à éviter absolument.", "négatif"),
    ("La batterie se vide en une heure.", "négatif"),
    ("Service client injoignable, catastrophe.", "négatif"),
    ("Trop cher pour ce que c'est.", "négatif"),
    ("Le produit est tombé en panne au bout d'un jour.", "négatif"),
    ("Emballage abîmé et pièce manquante.", "négatif"),
    ("Je regrette cet achat, très mauvaise expérience.", "négatif"),
    ("Livraison en retard de deux semaines.", "négatif"),
    ("Bruyant et inconfortable à l'usage.", "négatif"),
    ("Aucune notice, impossible à configurer.", "négatif"),
    ("Franchement bluffé par la performance.", "positif"),
    ("Je rachèterai sans hésiter.", "positif"),
    ("Une arnaque, ne perdez pas votre argent.", "négatif"),
    ("Le meilleur achat de l'année.", "positif"),
    ("Produit défectueux dès la sortie du carton.", "négatif"),
    ("Confortable, joli et pas cher.", "positif"),
]
print(len(donnees), "exemples.")

30 exemples.


In [4]:
# Mise au format instruction -> réponse, puis découpage train / validation
from datasets import Dataset

exemples = [{"input": f"Classe le sentiment (positif ou négatif) : {t}", "output": lab}
            for (t, lab) in donnees]
random.shuffle(exemples)
train_ex, val_ex = exemples[:24], exemples[24:]
ds_train = Dataset.from_list(train_ex)
print("Train :", len(train_ex), "| Validation :", len(val_ex))

Train : 24 | Validation : 6


In [5]:
# Tokenisation (padding dynamique géré ensuite par le collator)
def preparer(ex):
    entree = tokenizer(ex["input"], max_length=64, truncation=True)
    cible  = tokenizer(text_target=ex["output"], max_length=6, truncation=True)
    entree["labels"] = cible["input_ids"]
    return entree

ds_train_tok = ds_train.map(preparer, remove_columns=ds_train.column_names)
print("Exemple tokenisé :", ds_train_tok[0])

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Exemple tokenisé : {'input_ids': [4501, 15, 90, 6493, 41, 19882, 17, 99, 407, 14563, 5497, 99, 61, 3, 10, 391, 13755, 3660, 18, 2246, 226, 256, 3697, 3869, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [28181, 1]}


## Étape 3 — Configurer LoRA
On **gèle** le modèle et on ajoute des petites matrices de rang faible sur les projections `q` et `v` de l'attention. Seules ces matrices seront entraînées.

In [12]:
!pip -q uninstall -y torchao

from peft import LoraConfig, get_peft_model, TaskType

config_lora = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,                 # rang des matrices (petit = léger)
    lora_alpha=16,       # facteur d'échelle
    lora_dropout=0.05,
    target_modules=["q", "v"],   # où insérer LoRA (attention)
)
modele = get_peft_model(modele, config_lora)
modele.print_trainable_parameters()   # <-- observez le pourcentage entraîné

trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


**À observer** : `print_trainable_parameters()` affiche par exemple *trainable params ≈ 0,3 %*. C'est tout l'intérêt de LoRA : adapter le modèle en n'entraînant qu'une infime partie des poids.

## Étape 4 — Entraîner

In [13]:
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)

collator = DataCollatorForSeq2Seq(tokenizer, model=modele)

args = Seq2SeqTrainingArguments(
    output_dir="lora-sentiment",
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    num_train_epochs=30,
    logging_steps=5,
    report_to="none",
)

trainer = Seq2SeqTrainer(model=modele, args=args,
                         train_dataset=ds_train_tok, data_collator=collator)
trainer.train()

Step,Training Loss
5,4.231542
10,2.457557
15,1.101581
20,0.748231
25,0.407920
30,0.389293
35,0.328742
40,0.392365
45,0.233755
50,0.263559


TrainOutput(global_step=90, training_loss=0.6664129349920485, metrics={'train_runtime': 28.0825, 'train_samples_per_second': 25.639, 'train_steps_per_second': 3.205, 'total_flos': 8549925666816.0, 'train_loss': 0.6664129349920485, 'epoch': 30.0})

**À observer** : la *training loss* (perte) doit **descendre** au fil des époques. Si elle stagne haut, augmentez `num_train_epochs` ou `learning_rate`.

## Étape 5 — Comparer avant / après

In [14]:
print("=== APRÈS fine-tuning ===")
for t in tests:
    print(f"  {t[:45]:45} -> {predire(t)}")

print("\n=== Sur la validation (exemples jamais vus à l'entraînement) ===")
bons = 0
for ex in val_ex:
    texte = ex["input"].split(": ", 1)[1]
    pred = predire(texte).lower()
    attendu = ex["output"]
    ok = attendu in pred
    bons += ok
    print(f"  attendu={attendu:8} | prédit={pred:10} | {'OK' if ok else 'X'}  ({texte[:40]})")
print(f"\nExactitude validation : {bons}/{len(val_ex)}")

=== APRÈS fine-tuning ===
  Ce produit est vraiment excellent, je recomma -> positif
  Très déçu, l'appareil est tombé en panne en d -> négatif

=== Sur la validation (exemples jamais vus à l'entraînement) ===
  attendu=positif  | prédit=positif    | OK  (Très bon produit, je recommande vivement)
  attendu=positif  | prédit=négatif    | X  (Livraison rapide et emballage soigné.)
  attendu=négatif  | prédit=positif    | X  (Ne fonctionne pas comme annoncé.)
  attendu=positif  | prédit=négatif    | X  (Franchement bluffé par la performance.)
  attendu=négatif  | prédit=positif    | X  (Déçu, l'article est arrivé cassé.)
  attendu=positif  | prédit=positif    | OK  (Le meilleur achat de l'année.)

Exactitude validation : 2/6


In [17]:
!pip -q install -U langchain langgraph langchain-google-genai

import os, re
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")  # lu depuis les Secrets, jamais en clair

from langchain.agents import create_agent               # nouvel emplacement (LangGraph V1)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool


# --- Outil 1 : calculatrice sécurisée ---
@tool
def calc(expr: str) -> str:
    """Évalue une expression arithmétique simple (ex. '3*12-5')."""
    if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s]+", expr):
        return "Expression non autorisée."
    return str(eval(expr, {"_builtins_": {}}, {}))


# --- Outil 2 : recherche dans une petite base interne ---
@tool
def recherche(requete: str) -> str:
    """Cherche un fait dans une petite base de connaissances interne."""
    base = {
        "capitale france": "La capitale de la France est Paris.",
        "tour eiffel": "La tour Eiffel a été achevée en 1889.",
        "mont blanc": "Le mont Blanc culmine à 4809 mètres.",
    }
    req = requete.lower()
    for cle, valeur in base.items():
        if any(mot in req for mot in cle.split()):
            return valeur
    return "Aucun résultat."


# --- L'agent ReAct ---
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)   # <-- modèle valide (ta liste)
agent = create_agent(llm, tools=[calc, recherche])


def demander_agent(question):
    resultat = agent.invoke({"messages": [("user", question)]})
    return resultat["messages"][-1].content


# --- Tests ---
print("Q1 :", demander_agent("Combien font 3*12-5 ?"))
print("Q2 :", demander_agent("Quelle est la hauteur du mont Blanc ?"))
print("Q3 :", demander_agent("Quelle est la hauteur du mont JACQUES ?"))
print("Q4 :", demander_agent("Quelle est la hauteur du mont?"))

Q1 : 3*12-5 font 31.
Q2 : [{'type': 'text', 'text': 'Le mont Blanc culmine à 4809 mètres.', 'extras': {'signature': 'Cv8BARFNMg9XRmCAk5S4ZlHICyNwQyqQFNlbMweSuJwSKlp1nC61bJ3/E9VzP/Dn8B8uABy/TfZGip82yYAPRMamFXYCy9ZBi7n39D2csycHjXQtqffu1Jk0NBvEHtLpzBltgtIInxXlVuTkM2pc/YscqTf1o0Hx7gMPeGaagdb+LHsr/qZeVeYdZCOkwx4WHIllQz75O0aJNVL0VCi6tovtZPcAzZQuBIyp0pZKtWu7eGGS5hZo36TzhgM/CPzWH2Iqtf+ls5JAsx50yoaRfo3kNOH00k1FdUApIzwhZl4oNl6huIpznOyElhTb/UE60NKfSTk7YIn2V3pGgc9MptXq'}}]
Q3 : [{'type': 'text', 'text': "Je n'ai pas trouvé d'information sur la hauteur du mont JACQUES.", 'extras': {'signature': 'Co8EARFNMg9LN/yN1xtvCJgnk+GFOymhp4gyadiWnETmOXr8kdOmBeqAvZd7tUrAhzPWuxOyjrikjiCZtTQ8RkUe5chr4O1blSLE2lyuPtVN9qrB45tvHwHjrdIJO7zGDHLYsQu6C7QAGCcZOwByb7rYEWHmNLDfsxTOq2Nh0S0EpwfLsnlgknS/Agbqr5Qv0UkJn+TuQ6a/U7CUfkLXjl8jahjgg2iBQHHLKPRWCH98hW2kwwZ7/Eb3yOeeNGD547lnsGTToP8c7hDt1efXesFXU/5HfUIkJRSKI/d3dd6IvQThNDWL0nezhA0IrNKSFyAA/QFZNkWFYrkJh3xGpPgEbdyZIL8be9l+EKg6utFsidY365NnT0lAzb34gRc8IoJH1B0LlDJ1NJG1M/arV/MMK3

**À observer** : après entraînement, le modèle répond « positif » / « négatif » de façon cohérente en français — comportement qu'il n'avait pas avant. C'est un **changement de comportement**, obtenu en entraînant ~0,3 % des paramètres.

### Exercices proposés
1. Faites varier `r` (2, 8, 16) et `num_train_epochs` : quel effet sur l'exactitude ?
2. Ajoutez une 3e classe « neutre » avec quelques exemples et ré-entraînez.
3. Surveillez le **sur-apprentissage** : si la training loss est très basse mais la validation mauvaise, réduisez les époques.


## Bilan du TP 2
- **LoRA** ajoute des matrices de **rang faible** ; on n'entraîne qu'une fraction des poids.
- On a transformé un modèle généraliste en **classifieur de sentiment français**.
- Toujours **évaluer sur des données non vues** et surveiller l'overfitting.

➡️ **TP 3** : on donne au modèle la capacité d'**agir** avec des outils (agent).
